# Minimal Colab Pipeline for Playwright run_spec Dataset

This notebook validates the ML pipeline only.

It does **not**:
- call Laravel
- touch the database
- launch Playwright
- modify the application

It only:
- loads `colab_train.jsonl` and `colab_eval.jsonl`
- builds prompts from `instruction + input`
- builds targets from `output`
- runs a lightweight Hugging Face instruct model
- checks simple JSON-oriented metrics

This is a **prototype notebook** for pipeline validation, not real training.

In [ ]:
!pip -q install transformers datasets accelerate sentencepiece

In [ ]:
import json
from pathlib import Path

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

## Data Loading

You have two options:
1. upload the files manually in Colab
2. place them in `/content/`

Expected files:
- `colab_train.jsonl`
- `colab_eval.jsonl`

In [ ]:
if IN_COLAB:
    print('If files are not already in /content, upload them now.')
    uploaded = files.upload()
else:
    print('Running outside Colab; expecting local files.')

In [ ]:
TRAIN_PATH = Path('/content/colab_train.jsonl') if IN_COLAB else Path('backend/storage/app/agent-training/colab/colab_train.jsonl')
EVAL_PATH = Path('/content/colab_eval.jsonl') if IN_COLAB else Path('backend/storage/app/agent-training/colab/colab_eval.jsonl')

def load_jsonl(path: Path):
    records = []
    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

train_records = load_jsonl(TRAIN_PATH)
eval_records = load_jsonl(EVAL_PATH)

print('Train examples:', len(train_records))
print('Eval examples:', len(eval_records))
if len(train_records) + len(eval_records) < 50:
    print('Warning: Dataset too small for real training; use only for pipeline prototype.')

In [ ]:
for i, example in enumerate(train_records[:3], start=1):
    print(f'Example {i}')
    print(json.dumps(example, indent=2, ensure_ascii=False))
    print('-' * 80)

## Prompt / Target Construction

In [ ]:
def build_prompt(example):
    instruction = example['instruction']
    input_payload = json.dumps(example['input'], ensure_ascii=False, indent=2)
    return f"{instruction}\n\nInput:\n{input_payload}\n\nOutput JSON:" 

def build_target(example):
    return json.dumps(example['output'], ensure_ascii=False)

print(build_prompt(train_records[0]))
print('\nExpected target:')
print(build_target(train_records[0]))

In [ ]:
train_dataset = Dataset.from_list([
    {
        'prompt': build_prompt(example),
        'target': build_target(example),
        'metadata': example['metadata'],
    }
    for example in train_records
])

eval_dataset = Dataset.from_list([
    {
        'prompt': build_prompt(example),
        'target': build_target(example),
        'metadata': example['metadata'],
    }
    for example in eval_records
])

print(train_dataset)
print(eval_dataset)

## Lightweight Hugging Face Model

For a minimal prototype, we use `google/flan-t5-small`.
This is not expected to perform well on such a small dataset.
It is only used to validate the pipeline end-to-end.

In [ ]:
MODEL_NAME = 'google/flan-t5-small'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print('Loaded model:', MODEL_NAME)

In [ ]:
sample_eval = eval_records[0]
sample_prompt = build_prompt(sample_eval)

inputs = tokenizer(sample_prompt, return_tensors='pt', truncation=True, max_length=1024)
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False,
)
prediction_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print('Prompt:')
print(sample_prompt)
print('\nPrediction:')
print(prediction_text)
print('\nExpected:')
print(build_target(sample_eval))

In [ ]:
def parse_json_or_none(text):
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        start = text.find('{')
        end = text.rfind('}')
        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(text[start:end+1])
            except Exception:
                return None
        return None

parsed_prediction = parse_json_or_none(prediction_text)
print('Valid JSON:', parsed_prediction is not None)
if parsed_prediction is not None:
    print(json.dumps(parsed_prediction, indent=2, ensure_ascii=False))

## Simple Eval Metrics

We compute very lightweight prototype metrics on the eval set:
- `valid_json_rate`
- `exact_scenario_type_match`
- `contains_steps`
- `contains_asserts`

This is not a real benchmark. It is only a pipeline sanity check.

In [ ]:
def generate_output(prompt):
    model_inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024)
    model_outputs = model.generate(
        **model_inputs,
        max_new_tokens=256,
        do_sample=False,
    )
    return tokenizer.decode(model_outputs[0], skip_special_tokens=True)

results = []
for example in eval_records:
    prompt = build_prompt(example)
    raw_prediction = generate_output(prompt)
    parsed = parse_json_or_none(raw_prediction)
    target = example['output']

    valid_json = parsed is not None
    exact_scenario_type_match = False
    contains_steps = False
    contains_asserts = False

    if valid_json and isinstance(parsed, dict):
        exact_scenario_type_match = parsed.get('scenario_type') == target.get('scenario_type')
        contains_steps = isinstance(parsed.get('steps'), list) and len(parsed.get('steps', [])) > 0
        contains_asserts = isinstance(parsed.get('asserts'), list) and len(parsed.get('asserts', [])) > 0

    results.append({
        'valid_json': valid_json,
        'exact_scenario_type_match': exact_scenario_type_match,
        'contains_steps': contains_steps,
        'contains_asserts': contains_asserts,
        'prediction': raw_prediction,
        'expected_scenario_type': target.get('scenario_type'),
    })

total = len(results)
metrics = {
    'valid_json_rate': sum(item['valid_json'] for item in results) / total if total else 0.0,
    'exact_scenario_type_match': sum(item['exact_scenario_type_match'] for item in results) / total if total else 0.0,
    'contains_steps': sum(item['contains_steps'] for item in results) / total if total else 0.0,
    'contains_asserts': sum(item['contains_asserts'] for item in results) / total if total else 0.0,
}

print(json.dumps(metrics, indent=2))

In [ ]:
for item in results[:3]:
    print('Expected scenario_type:', item['expected_scenario_type'])
    print('Valid JSON:', item['valid_json'])
    print('Prediction:')
    print(item['prediction'])
    print('-' * 80)

## Limits

- The dataset is very small.
- This notebook is only for pipeline validation.
- Do not interpret results as real model quality.
- A meaningful fine-tuning phase would require many more examples and better coverage.
- The next goal is dataset growth and stronger scenario diversity, not performance claims.